In [1]:
# ==========================================
# STEP 1 — Install Required Libraries
# ==========================================

!pip install pdfplumber spacy transformers torch -q

# Download Spacy Model
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# ==========================================
# STEP 2 — Upload PDF File in Google Colab
# ==========================================

from google.colab import files

uploaded = files.upload()

Saving loksabha gvmt.bill.pdf to loksabha gvmt.bill (1).pdf


In [3]:
# ==========================================
# STEP 3 — Get Uploaded PDF File Name
# ==========================================

pdf_file = list(uploaded.keys())[0]

print("Uploaded File:", pdf_file)

Uploaded File: loksabha gvmt.bill (1).pdf


In [4]:
# ==========================================
# STEP 4 — Import Libraries
# ==========================================

import pdfplumber
import re
import json
import spacy
from transformers import pipeline

In [5]:
# ==========================================
# STEP 5 — Load NLP Models
# ==========================================

import spacy

nlp = spacy.load("en_core_web_sm")


In [8]:
# ==========================================
# STEP 6 — Extract Text from Uploaded PDF
# ==========================================

def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


# Ensure the pdf_file exists on disk before proceeding
# This step addresses cases where the runtime restarts and uploaded files are lost
if pdf_file in uploaded:
    with open(pdf_file, 'wb') as f:
        f.write(uploaded[pdf_file])

raw_text = extract_text_from_pdf(pdf_file)

print(raw_text[:2000])

AS INTRODUCED IN LOK SABHA
Bill No. 113 of 2023
THE DIGITAL PERSONAL DATA PROTECTION BILL, 2023
——————
ARRANGEMENT OF CLAUSES
——————
CHAPTER I
PRELIMINARY
CLAUSES
1. Short title and commencement.
2. Definitions.
3. Application of Act.
CHAPTER II
OBLIGATIONS OF DATA FIDUCIARY
4. Grounds for processing personal data.
5. Notice.
6. Consent.
7. Certain legitimate uses.
8. General obligations of Data Fiduciary.
9. Processing of personal data of children.
10. Additional obligations of Significant Data Fiduciary.
CHAPTER III
RIGHTS AND DUTIES OF DATA PRINCIPAL
11. Right to access information about personal data.
12. Right to correction and erasure of personal data.
13. Right of grievance redressal.
14. Right to nominate.
15. Duties of Data Principal.
CHAPTER IV
SPECIAL PROVISIONS
16. Processing of personal data outside India.
17. Exemptions.
CHAPTER V
DATA PROTECTION BOARD OF INDIA
18. Establishment of Board.
19. Composition and qualifications for appointment of Chairperson and Members.
20. S

In [9]:
# ==========================================
# STEP 7 — Clean Extracted Text
# ==========================================

def clean_text(text):

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'Page \d+', '', text)

    text = re.sub(r'[^a-zA-Z0-9.,;:()\-\s]', '', text)

    return text.strip()


clean_data = clean_text(raw_text)

print(clean_data[:2000])

AS INTRODUCED IN LOK SABHA Bill No. 113 of 2023 THE DIGITAL PERSONAL DATA PROTECTION BILL, 2023  ARRANGEMENT OF CLAUSES  CHAPTER I PRELIMINARY CLAUSES 1. Short title and commencement. 2. Definitions. 3. Application of Act. CHAPTER II OBLIGATIONS OF DATA FIDUCIARY 4. Grounds for processing personal data. 5. Notice. 6. Consent. 7. Certain legitimate uses. 8. General obligations of Data Fiduciary. 9. Processing of personal data of children. 10. Additional obligations of Significant Data Fiduciary. CHAPTER III RIGHTS AND DUTIES OF DATA PRINCIPAL 11. Right to access information about personal data. 12. Right to correction and erasure of personal data. 13. Right of grievance redressal. 14. Right to nominate. 15. Duties of Data Principal. CHAPTER IV SPECIAL PROVISIONS 16. Processing of personal data outside India. 17. Exemptions. CHAPTER V DATA PROTECTION BOARD OF INDIA 18. Establishment of Board. 19. Composition and qualifications for appointment of Chairperson and Members. 20. Salary, allow

In [10]:
# ==========================================
# STEP 8 — Extract Bill Information
# ==========================================

# ---------- Title ----------
def extract_title(text):

    lines = text.split(".")

    for line in lines[:15]:

        if "bill" in line.lower():
            return line.strip()

    return "Title Not Found"


# ---------- Ministry ----------
def extract_ministry(text):

    ministries = [
        "Ministry of Finance",
        "Ministry of Electronics and Information Technology",
        "Ministry of Home Affairs",
        "Ministry of Law and Justice",
        "Ministry of Health"
    ]

    for ministry in ministries:

        if ministry.lower() in text.lower():
            return ministry

    return "Ministry Not Found"


# ---------- Objective ----------
def extract_objective(text):

    sentences = text.split(".")

    keywords = [
        "objective",
        "purpose",
        "aim",
        "bill seeks to",
        "an act to"
    ]

    for sentence in sentences:

        for keyword in keywords:

            if keyword in sentence.lower():
                return sentence.strip()

    return "Objective Not Found"


# ---------- Key Provisions ----------
def extract_key_provisions(text):

    provisions = []

    sentences = text.split(".")

    keywords = [
        "shall",
        "penalty",
        "rights",
        "authority",
        "regulation"
    ]

    for sentence in sentences:

        for keyword in keywords:

            if keyword in sentence.lower():
                provisions.append(sentence.strip())
                break

    return provisions[:10]


# ---------- Related Acts ----------
def extract_related_acts(text):

    pattern = r'([A-Z][A-Za-z\s]+Act,\s\d{4})'

    acts = re.findall(pattern, text)

    return list(set(acts))


# ---------- Stakeholders ----------
def extract_stakeholders(text):

    doc = nlp(text)

    stakeholders = []

    for ent in doc.ents:

        if ent.label_ in ["ORG", "PERSON", "GPE"]:
            stakeholders.append(ent.text)

    return list(set(stakeholders))[:20]


# ---------- Industries ----------
def extract_industries(text):

    industries = [
        "IT",
        "Healthcare",
        "Finance",
        "Education",
        "Agriculture",
        "Telecom"
    ]

    found = []

    for industry in industries:

        if industry.lower() in text.lower():
            found.append(industry)

    return found


# ---------- Risks & Opportunities ----------
def extract_risks_opportunities(text):

    risks = []

    opportunities = []

    sentences = text.split(".")

    for sentence in sentences:

        if any(word in sentence.lower() for word in
               ["risk", "penalty", "burden", "challenge"]):

            risks.append(sentence.strip())

        if any(word in sentence.lower() for word in
               ["benefit", "growth", "opportunity", "improvement"]):

            opportunities.append(sentence.strip())

    return {
        "risks": risks[:5],
        "opportunities": opportunities[:5]
    }

In [11]:
# ==========================================
# STEP 9 — Generate Simple Summary
# ==========================================

def generate_summary(text, num_sentences=5):

    sentences = text.split(".")

    summary = ".".join(sentences[:num_sentences])

    return summary


summary_text = generate_summary(clean_data)

print(summary_text)

AS INTRODUCED IN LOK SABHA Bill No. 113 of 2023 THE DIGITAL PERSONAL DATA PROTECTION BILL, 2023  ARRANGEMENT OF CLAUSES  CHAPTER I PRELIMINARY CLAUSES 1. Short title and commencement. 2. Definitions


In [12]:
# ==========================================
# STEP 10 — Create Final Structured JSON
# ==========================================

final_output = {

    "title": extract_title(clean_data),

    "ministry": extract_ministry(clean_data),

    "objective": extract_objective(clean_data),

    "summary": summary_text,

    "key_provisions": extract_key_provisions(clean_data),

    "related_previous_bills_acts":
        extract_related_acts(clean_data),

    "stakeholders":
        extract_stakeholders(clean_data),

    "affected_industries":
        extract_industries(clean_data),

    "risks_opportunities":
        extract_risks_opportunities(clean_data)
}

print(json.dumps(final_output, indent=4))

{
    "title": "AS INTRODUCED IN LOK SABHA Bill No",
    "ministry": "Ministry Not Found",
    "objective": "113 of 2023 THE DIGITAL PERSONAL DATA PROTECTION BILL, 2023 A BILL to provide for the processing of digital personal data in a manner that recognises both the right of individuals to protect their personal data and the need to process such personal data for lawful purposes and for matters connected therewith or incidental thereto",
    "summary": "AS INTRODUCED IN LOK SABHA Bill No. 113 of 2023 THE DIGITAL PERSONAL DATA PROTECTION BILL, 2023  ARRANGEMENT OF CLAUSES  CHAPTER I PRELIMINARY CLAUSES 1. Short title and commencement. 2. Definitions",
    "key_provisions": [
        "CHAPTER III RIGHTS AND DUTIES OF DATA PRINCIPAL 11",
        "(2) It shall come into force on such date as the Central Government may, by notification in the Official Gazette, appoint and different dates may be appointed for different provisions of this Act and any reference in any such provision to the co

In [32]:
# ==========================================
# STEP 11 — Save JSON Output File
# ==========================================

!pip install pdfplumber -q

import json
import re
import spacy
import pdfplumber

# Ensure nlp model is loaded if not already in scope
try:
    nlp
except NameError:
    nlp = spacy.load("en_core_web_sm")

# Re-define functions if not in scope (copied from STEP 8 and STEP 9)
# ---------- Title ----------
def extract_title(text):
    lines = text.split(".")
    for line in lines[:15]:
        if "bill" in line.lower():
            return line.strip()
    return "Title Not Found"

# ---------- Ministry ----------
def extract_ministry(text):
    ministries = [
        "Ministry of Finance",
        "Ministry of Electronics and Information Technology",
        "Ministry of Home Affairs",
        "Ministry of Law and Justice",
        "Ministry of Health"
    ]
    for ministry in ministries:
        if ministry.lower() in text.lower():
            return ministry
    return "Ministry Not Found"

# ---------- Objective ----------
def extract_objective(text):
    sentences = text.split(".")
    keywords = [
        "objective",
        "purpose",
        "aim",
        "bill seeks to",
        "an act to"
    ]
    for sentence in sentences:
        for keyword in keywords:
            if keyword in sentence.lower():
                return sentence.strip()
    return "Objective Not Found"

# ---------- Key Provisions ----------
def extract_key_provisions(text):
    provisions = []
    sentences = text.split(".")
    keywords = [
        "shall",
        "penalty",
        "rights",
        "authority",
        "regulation"
    ]
    for sentence in sentences:
        for keyword in keywords:
            if keyword in sentence.lower():
                provisions.append(sentence.strip())
                break
    return provisions[:10]

# ---------- Related Acts ----------
def extract_related_acts(text):
    pattern = r'([A-Z][A-Za-z\s]+Act,\s\d{4})'
    acts = re.findall(pattern, text)
    return list(set(acts))

# ---------- Stakeholders ----------
def extract_stakeholders(text):
    doc = nlp(text)
    stakeholders = []
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PERSON", "GPE"]:
            stakeholders.append(ent.text)
    return list(set(stakeholders))[:20]

# ---------- Industries ----------
def extract_industries(text):
    industries = [
        "IT",
        "Healthcare",
        "Finance",
        "Education",
        "Agriculture",
        "Telecom"
    ]
    found = []
    for industry in industries:
        if industry.lower() in text.lower():
            found.append(industry)
    return found

# ---------- Risks & Opportunities ----------
def extract_risks_opportunities(text):
    risks = []
    opportunities = []
    sentences = text.split(".")
    for sentence in sentences:
        if any(word in sentence.lower() for word in
               ["risk", "penalty", "burden", "challenge"]):
            risks.append(sentence.strip())
        if any(word in sentence.lower() for word in
               ["benefit", "growth", "opportunity", "improvement"]):
            opportunities.append(sentence.strip())
    return {
        "risks": risks[:5],
        "opportunities": opportunities[:5]
    }

# ---------- Generate Summary ----------
def generate_summary(text, num_sentences=5):
    sentences = text.split(".")
    summary = ".".join(sentences[:num_sentences])
    return summary

# ---------- Text Extraction (copied from STEP 6) ----------
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

# ---------- Text Cleaning (copied from STEP 7) ----------
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+', '', text)
    text = re.sub(r'[^a-zA-Z0-9.,;:()\-\s]', '', text)
    return text.strip()


# Reconstruct variables if they are not defined
pdf_file = "loksabha gvmt.bill.pdf" # Assuming this was the file uploaded previously

try:
    raw_text
except NameError:
    raw_text = extract_text_from_pdf(pdf_file)

try:
    clean_data
except NameError:
    clean_data = clean_text(raw_text)

try:
    summary_text
except NameError:
    summary_text = generate_summary(clean_data)


# Re-defining final_output here to ensure it's available
# This block was previously in STEP 10 (cell g9gtq0_Np7o3)
final_output = {
    "title": extract_title(clean_data),
    "ministry": extract_ministry(clean_data),
    "objective": extract_objective(clean_data),
    "summary": summary_text,
    "key_provisions": extract_key_provisions(clean_data),
    "related_previous_bills_acts": extract_related_acts(clean_data),
    "stakeholders": extract_stakeholders(clean_data),
    "affected_industries": extract_industries(clean_data),
    "risks_opportunities": extract_risks_opportunities(clean_data)
}

with open("bill_analysis.json", "w") as file:
    json.dump(final_output, file, indent=4)

print("JSON File Saved Successfully")

JSON File Saved Successfully


In [33]:
# ==========================================
# STEP 12 — Download JSON File
# ==========================================

from google.colab import files

files.download("bill_analysis.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
!pip install streamlit requests fpdf -q

In [41]:
%%writefile app.py

import streamlit as st
import requests
import json
from fpdf import FPDF
import pandas as pd

# ------------------------------------
# Streamlit UI
# ------------------------------------

st.set_page_config(
    page_title="AI Bill Analyzer",
    layout="wide"
)

st.title("🏛️ AI Parliament Bill Analyzer")

# ------------------------------------
# Upload PDF
# ------------------------------------

uploaded_file = st.file_uploader(
    "Upload Bill PDF",
    type=["pdf"]
)

# ------------------------------------
# Paste URL
# ------------------------------------

bill_url = st.text_input(
    "Or Paste Bill URL"
)

# ------------------------------------
# Process Button
# ------------------------------------

if st.button("Analyze Bill"):

    with st.spinner("Processing Bill..."):

        # --------------------------------
        # Load `final_output` from JSON file
        # --------------------------------
        try:
            with open("bill_analysis.json", "r") as f:
                final_output_data = json.load(f)
        except FileNotFoundError:
            st.error("Error: bill_analysis.json not found. Please run previous steps.")
            final_output_data = {}

        data = {
            "title": final_output_data.get("title", "N/A"),
            "ministry": final_output_data.get("ministry", "N/A"),
            "objective": final_output_data.get("objective", "N/A"),
            "summary": final_output_data.get("summary", "N/A"),
            "timeline": [
                "Not available in current extraction",
            ],
            "industries": {item: 100 for item in final_output_data.get("affected_industries", []) if item} if final_output_data.get("affected_industries") else {},
            "risks": final_output_data.get("risks_opportunities", {}).get("risks", []),
            "opportunities": final_output_data.get("risks_opportunities", {}).get("opportunities", []),
            "key_provisions": final_output_data.get("key_provisions", []),
            "stakeholders": final_output_data.get("stakeholders", []),
            "related_previous_bills_acts": final_output_data.get("related_previous_bills_acts", [])
        }

        # --------------------------------
        # Display Summary
        # --------------------------------

        st.header("📄 AI Summary")

        # Align summary text using markdown with HTML
        st.markdown(f"<div style='text-align: justify;'>{data['summary']}</div>", unsafe_allow_html=True)

        # --------------------------------
        # Bill Information
        # --------------------------------

        col1, col2 = st.columns(2)

        with col1:
            st.metric("Bill Title", data["title"])

        with col2:
            st.metric("Ministry", data["ministry"])

        # --------------------------------
        # Key Provisions
        # --------------------------------
        st.header("🔑 Key Provisions")
        if data["key_provisions"]:
            for provision in data["key_provisions"]:
                st.markdown(f"- {provision}")
        else:
            st.write("No key provisions found.")

        # --------------------------------
        # Related Acts
        # --------------------------------
        st.header("📜 Related Acts/Bills")
        if data["related_previous_bills_acts"]:
            for act in data["related_previous_bills_acts"]:
                st.markdown(f"- {act}")
        else:
            st.write("No related acts or bills found.")

        # --------------------------------
        # Stakeholders
        # --------------------------------
        st.header("👥 Stakeholders")
        if data["stakeholders"]:
            for stakeholder in data["stakeholders"]:
                st.markdown(f"- {stakeholder}")
        else:
            st.write("No stakeholders found.")

        # --------------------------------
        # Timeline View (Placeholder)
        # --------------------------------

        st.header("📅 Bill Timeline")

        for event in data["timeline"]:
            st.markdown(f"- {event}")

        # --------------------------------
        # Industry Impact
        # --------------------------------

        st.header("📊 Industry Impact")

        if data["industries"]:
            df = pd.DataFrame({
                "Industry": list(data["industries"].keys()),
                "Impact Score": list(data["industries"].values())
            })

            st.bar_chart(
                df.set_index("Industry")
            )
        else:
            st.write("No affected industries found.")

        # --------------------------------
        # Risks and Opportunities
        # --------------------------------

        col3, col4 = st.columns(2)

        with col3:

            st.subheader("⚠️ Risks")

            if data["risks"]:
                for risk in data["risks"]:
                    st.write("•", risk)
            else:
                st.write("No risks identified.")

        with col4:

            st.subheader("✅ Opportunities")

            if data["opportunities"]:
                for opp in data["opportunities"]:
                    st.write("•", opp)
            else:
                st.write("No opportunities identified.")

        # --------------------------------
        # Download PDF Summary
        # --------------------------------

        pdf = FPDF()

        pdf.add_page()

        pdf.set_font("Arial", size=12)
        pdf.multi_cell(0, 10, txt="AI Bill Analysis Report", align='C')
        pdf.ln(10)

        # Add Title
        pdf.set_font("Arial", 'B', 14)
        pdf.multi_cell(0, 10, txt=f"Bill Title: {data['title']}")
        pdf.ln(5)

        # Add Ministry
        pdf.set_font("Arial", 'B', 12)
        pdf.multi_cell(0, 10, txt=f"Ministry: {data['ministry']}")
        pdf.ln(5)

        # Add Objective
        pdf.multi_cell(0, 10, txt=f"Objective: {data['objective']}")
        pdf.ln(5)

        # Add Summary
        pdf.multi_cell(0, 10, txt=f"Summary: {data['summary']}")
        pdf.ln(10)

        # Add Key Provisions
        if data["key_provisions"]:
            pdf.set_font("Arial", 'B', 12)
            pdf.multi_cell(0, 10, txt="Key Provisions:")
            pdf.set_font("Arial", '', 10)
            for provision in data["key_provisions"]:
                pdf.multi_cell(0, 7, txt=f"- {provision}")
            pdf.ln(5)

        # Add Related Acts/Bills
        if data["related_previous_bills_acts"]:
            pdf.set_font("Arial", 'B', 12)
            pdf.multi_cell(0, 10, txt="Related Acts/Bills:")
            pdf.set_font("Arial", '', 10)
            for act in data["related_previous_bills_acts"]:
                pdf.multi_cell(0, 7, txt=f"- {act}")
            pdf.ln(5)

        # Add Stakeholders
        if data["stakeholders"]:
            pdf.set_font("Arial", 'B', 12)
            pdf.multi_cell(0, 10, txt="Stakeholders:")
            pdf.set_font("Arial", '', 10)
            for stakeholder in data["stakeholders"]:
                pdf.multi_cell(0, 7, txt=f"- {stakeholder}")
            pdf.ln(5)

        # Add Risks
        if data["risks"]:
            pdf.set_font("Arial", 'B', 12)
            pdf.multi_cell(0, 10, txt="Risks:")
            pdf.set_font("Arial", '', 10)
            for risk in data["risks"]:
                pdf.multi_cell(0, 7, txt=f"- {risk}")
            pdf.ln(5)

        # Add Opportunities
        if data["opportunities"]:
            pdf.set_font("Arial", 'B', 12)
            pdf.multi_cell(0, 10, txt="Opportunities:")
            pdf.set_font("Arial", '', 10)
            for opp in data["opportunities"]:
                pdf.multi_cell(0, 7, txt=f"- {opp}")
            pdf.ln(5)


        pdf.output("bill_summary.pdf")

        with open("bill_summary.pdf", "rb") as file:

            st.download_button(
                label="⬇ Download PDF Report",
                data=file,
                file_name="bill_summary.pdf",
                mime="application/pdf"
            )

Overwriting app.py


In [42]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [43]:
# Restart Streamlit application
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

# Re-establish Cloudflare tunnel
!cloudflared tunnel --url http://localhost:8501

2026-05-26T08:47:24Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-26T08:47:24Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-26T08:47:27Z INF +--------------------------------------------------------------------------------------------+
2026-05-26T08:47:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-26T08:47:27Z INF |  https://replication-theme-despite-volumes.trycloudfla

In [24]:
!cloudflared tunnel --url http://localhost:8501

2026-05-26T08:32:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-26T08:32:47Z INF Requesting new quick Tunnel on trycloudflare.com...
failed to request quick Tunnel: Post "https://api.trycloudflare.com/tunnel": context deadline exceeded (Client.Timeout exceeded while awaiting headers)
